imports

In [1]:
import torch

helper function to check if disterbution is values are positive and sum is 1

In [2]:
def check_if_dist_is_valid(dist: torch.Tensor) -> bool:
    if not torch.all(dist > 0):
        return False
    if not torch.isclose(torch.sum(dist), torch.tensor(1.0, dtype=dist.dtype)):
        return False
    return True

my sampler function fills a tensor of values from a disterbution given by an input tensor by calculating random values from 0-1 \
 using cdf function we mark new values based on ranges \
example if we have distribution [0.2, 0.3, 0.5] then cdf is [0.2, 0.5, 1.0] \
 if we generate random value 0.35 it is between 0.2 and 0.5 so we mark it as 1 \
  if we generate random value 0.6 it is between 0.5 and 1.0 so we mark it as 2

In [3]:
def my_sampler(size, dist, requires_grad=False):
    dist = torch.tensor(dist, dtype=torch.float32)
    # check if distribution is valid (positive values and sum to 1)
    ok = check_if_dist_is_valid(dist)
    if not ok:
        raise ValueError("Distribution is not valid")
    #if size is one there isn't really much to choose from
    if dist.numel() == 1:
        return torch.zeros(size, dtype=torch.long)

    #calculate the cumulative distribution function (CDF)
    cdf = torch.cumsum(dist, dim=0).detach()
    #genreate uniform random values will laster change to distribution given by cdf
    universe = torch.rand(size)
    #generate zeros tensor to hold the results
    result = torch.zeros(size, dtype=torch.long)

    for i in range(len(dist) - 1):
        #calculate lower and upper bound of the distribution for the current index
        lower_bound = 0 if i == 0 else cdf[i - 1]
        upper_bound = cdf[i]
        #goes over all values in universe(the random values we generated)
        #and checks if they are in the cdf range and saves them in a boolean array
        mask = (universe >= lower_bound) & (universe < upper_bound)
        #for each true value in the mask value is within the range of cdf so change it to i
        result[mask] = i

    # last value case handle
    mask = universe >= cdf[-2]
    result[mask] = len(dist) - 1

    # if require_grad turn it on but only after all computations are done
    if requires_grad:
        result = result.float()
        result = result.clone().detach().requires_grad_(True)

    return result




In [4]:
class MyScalar:
    value: float
    immediate_derivative: float
    def __init__(self, value: float, immediate_derivative: float = 1.0, parent=None):
        self.value = value
        self.immediate_derivative = immediate_derivative
        self.parent = parent

sinusoidal functions

In [5]:
def sin(a: MyScalar) -> MyScalar:
    value = torch.sin(torch.tensor(a.value))
    immediate_derivative = torch.cos(torch.tensor(a.value))
    return MyScalar(value.item(), immediate_derivative.item(), parent=a)


def cos(a: MyScalar) -> MyScalar:
    value = torch.cos(torch.tensor(a.value))
    immediate_derivative = -torch.sin(torch.tensor(a.value))
    return MyScalar(value.item(), immediate_derivative.item(), parent=a)


basic arithmetic operations

In [6]:
def addition(a: MyScalar, b):
    value = a.value + b
    immediate_derivative = 1
    return MyScalar(value, immediate_derivative, parent=a)


def multiplication(a: MyScalar, b):
    value = a.value * b
    immediate_derivative = b
    return MyScalar(value, immediate_derivative, parent=a)


def power(a: MyScalar, b):
    value = a.value ** b
    immediate_derivative = b * (a.value ** (b - 1))
    return MyScalar(value, immediate_derivative, parent=a)

exponential and logarithmic functions

In [7]:
def exp(a: MyScalar) -> MyScalar:
    value = torch.exp(torch.tensor(a.value))
    immediate_derivative = value
    return MyScalar(value.item(), immediate_derivative.item(), parent=a)


def ln(a: MyScalar) -> MyScalar:
    value = torch.log(torch.tensor(a.value))
    immediate_derivative = 1 / a.value
    return MyScalar(value.item(), immediate_derivative, parent=a)

get gradient function calls recursively parent node (bottom up approach)\
 at each step it multiplies the accumulated gradient with the immediate derivative of the current node if we reach parentless node then we stop \
  the gradient dict is filled with value for each step each node is the key for the dict meaning after finishing the node

In [8]:
def get_gradient(scalar: MyScalar) -> dict:
    #define dict that calculates iteratively the gradient
    grad_dict = {}
    #helper recursive function
    def recursive_grad(node, accumulated_grad=1.0):
        grad_dict[node] = accumulated_grad
        node.operations = grad_dict
        if node.parent is not None:
            # recursive call with multiplication of gradient when there is a parent node
            recursive_grad(node.parent, accumulated_grad * node.immediate_derivative)

    #call for helper function
    recursive_grad(scalar)
    return grad_dict


tests for my_sampler

In [9]:
import numpy as np

# Test 1: Valid distribution, check output shape and value range
dist = [0.2, 0.3, 0.5]
sample_size = 10000
samples = my_sampler(sample_size, dist)
assert samples.shape[0] == sample_size, "Output shape mismatch"
assert torch.all((samples >= 0) & (samples < len(dist))), "Sample values out of range"

# Test 2: Empirical distribution matches input distribution (within tolerance)
counts = np.bincount(samples.numpy(), minlength=len(dist))
empirical = counts / sample_size
np.testing.assert_allclose(empirical, dist, atol=0.02)

# Test 3: Invalid distribution (contains zero)
try:
    my_sampler(10, [0.0, 0.5, 0.5])
    assert False, "Should have raised ValueError for zero in distribution"
except ValueError:
    pass

# Test 4: Invalid distribution (contains negative)
try:
    my_sampler(10, [0.2, -0.1, 0.9])
    assert False, "Should have raised ValueError for negative in distribution"
except ValueError:
    pass

# Test 5: Invalid distribution (does not sum to 1)
try:
    my_sampler(10, [0.2, 0.3, 0.6])
    assert False, "Should have raised ValueError for sum != 1"
except ValueError:
    pass

# Test 6: Single-element distribution
samples = my_sampler(10, [1.0])
assert torch.all(samples == 0), "All samples should be 0 for single-element distribution"

# Test 7: Size zero
samples = my_sampler(0, [1.0])
assert samples.shape[0] == 0, "Output should be empty for size=0"

# Test 8: Non-tensor input (list)
samples = my_sampler(100, [0.5, 0.5])
assert samples.shape[0] == 100, "Output shape mismatch for list input"

# Additional tests for my_sampler
import math

# Test 9: Large distribution
large_dist = np.ones(100) / 100
samples = my_sampler(10000, large_dist)
assert samples.shape[0] == 10000, "Output shape mismatch for large distribution"
assert torch.all((samples >= 0) & (samples < 100)), "Sample values out of range for large distribution"

# Test 10: Very small probabilities
small_dist = [1e-8, 1 - 1e-8]
samples = my_sampler(1000, small_dist)
assert torch.all((samples == 0) | (samples == 1)), "Sample values out of range for small probabilities"

# Test 11: Float32 and float64 input
dist32 = np.array([0.4, 0.6], dtype=np.float32)
dist64 = np.array([0.4, 0.6], dtype=np.float64)
samples32 = my_sampler(100, dist32)
samples64 = my_sampler(100, dist64)
assert samples32.shape[0] == 100 and samples64.shape[0] == 100, "Output shape mismatch for float32/64"

# Test 12: Numpy array input
dist_np = np.array([0.7, 0.3])
samples = my_sampler(100, dist_np)
assert samples.shape[0] == 100, "Output shape mismatch for numpy array input"

# Test 13: Torch tensor input
dist_tensor = torch.tensor([0.5, 0.5])
samples = my_sampler(100, dist_tensor)
assert samples.shape[0] == 100, "Output shape mismatch for torch tensor input"

# Test 14: Non-1D distribution (should raise error)
try:
    my_sampler(10, [[0.5, 0.5]])
    assert False, "Should have raised ValueError for non-1D distribution"
except Exception:
    pass

# Test 15: Distribution with NaN or inf (should raise error)
try:
    my_sampler(10, [float('nan'), 1.0])
    assert False, "Should have raised ValueError for NaN in distribution"
except Exception:
    pass
try:
    my_sampler(10, [float('inf'), 1.0])
    assert False, "Should have raised ValueError for inf in distribution"
except Exception:
    pass

# Test 16: Distribution sum very close to 1 (within tolerance)
dist = [0.333333, 0.333333, 0.333334]
samples = my_sampler(1000, dist)
assert samples.shape[0] == 1000, "Output shape mismatch for sum close to 1"

# Test 17: Negative sample size (should raise error)
try:
    my_sampler(-5, [0.5, 0.5])
    assert False, "Should have raised error for negative sample size"
except Exception:
    pass

# Test 18: requires_grad=True (should return tensor with requires_grad=True and float dtype)
samples = my_sampler(10, [0.5, 0.5], requires_grad=True)
assert samples.requires_grad, "Output should require grad when requires_grad=True"
assert samples.dtype == torch.float32, "Output should be float when requires_grad=True"

print("All tests passed.")


All tests passed.


C:\Users\asus\AppData\Local\Temp\ipykernel_49568\2484990078.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dist = torch.tensor(dist, dtype=torch.float32)


tests for MyScalar and gradient calculations

In [10]:
import torch

def torch_chain_exp_power(x):
    b = x ** 2      # b = a^2
    c = torch.exp(b)  # c = exp(a^2)
    return c

def torch_chain_sin_add(x):
    b = torch.sin(x)  # b = sin(a)
    c = b + 3         # c = sin(a) + 3
    return c

def torch_chain_cos_mul(x):
    b = torch.cos(x)  # b = cos(a)
    c = b * 4         # c = cos(a) * 4
    return c

def torch_chain_ln_add(x):
    b = torch.log(x)  # b = ln(a)
    c = b + 5         # c = ln(a) + 5
    return c

def torch_chain_power_add(x):
    b = x ** 3        # b = a^3
    c = b + 2         # c = a^3 + 2
    return c

def torch_chain_exp_mul(x):
    b = torch.exp(x)  # b = exp(a)
    c = b * 5         # c = exp(a) * 5
    return c

def torch_chain_sin_add_const(x):
    b = torch.sin(x)  # b = sin(a)
    c = b + 3         # c = sin(a) + 3
    return c

# --- Chained operations for MyScalar ---
def my_chain_exp_power(a):
    b = power(a, 2)
    c = exp(b)
    return c

def my_chain_sin_add(a):
    b = sin(a)
    c = addition(b, 3)
    return c

def my_chain_cos_mul(a):
    b = cos(a)
    c = multiplication(b, 4)
    return c

def my_chain_ln_add(a):
    b = ln(a)
    c = addition(b, 5)
    return c

def my_chain_power_add(a):
    b = power(a, 3)
    c = addition(b, 2)
    return c

def my_chain_exp_mul(a):
    b = exp(a)
    c = multiplication(b, 5)
    return c

def my_chain_sin_add_const(a):
    b = sin(a)
    c = addition(b, 3)
    return c

# --- Comparison function ---
def compare_chains(my_value, torch_func, my_func):
    # PyTorch
    x = torch.tensor(my_value, requires_grad=True, dtype=torch.float32)
    y_torch = torch_func(x)
    y_torch.backward()
    grad_torch = x.grad.item()

    # MyScalar
    a = MyScalar(my_value)
    y_my = my_func(a)
    grad_my = get_gradient(y_my)[a]

    print(f"Input: {my_value}")
    print(f"PyTorch value: {y_torch.item():.6f}, MyScalar value: {y_my.value:.6f}")
    print(f"PyTorch grad: {grad_torch:.6f}, MyScalar grad: {grad_my:.6f}")
    print("-"*50)

# --- Run all tests ---
compare_chains(2.0, torch_chain_exp_power, my_chain_exp_power)       # exp(a^2)
compare_chains(0.5, torch_chain_sin_add, my_chain_sin_add)           # sin(a)+3
compare_chains(0.25, torch_chain_cos_mul, my_chain_cos_mul)          # cos(a)*4
compare_chains(2.0, torch_chain_ln_add, my_chain_ln_add)             # ln(a)+5
compare_chains(3.0, torch_chain_power_add, my_chain_power_add)       # a^3+2
compare_chains(1.0, torch_chain_exp_mul, my_chain_exp_mul)           # exp(a)*5
compare_chains(0.7, torch_chain_sin_add_const, my_chain_sin_add_const)  # sin(a)+3



Input: 2.0
PyTorch value: 54.598148, MyScalar value: 54.598148
PyTorch grad: 218.392593, MyScalar grad: 218.392593
--------------------------------------------------
Input: 0.5
PyTorch value: 3.479425, MyScalar value: 3.479426
PyTorch grad: 0.877583, MyScalar grad: 0.877583
--------------------------------------------------
Input: 0.25
PyTorch value: 3.875650, MyScalar value: 3.875650
PyTorch grad: -0.989616, MyScalar grad: -0.989616
--------------------------------------------------
Input: 2.0
PyTorch value: 5.693147, MyScalar value: 5.693147
PyTorch grad: 0.500000, MyScalar grad: 0.500000
--------------------------------------------------
Input: 3.0
PyTorch value: 29.000000, MyScalar value: 29.000000
PyTorch grad: 27.000000, MyScalar grad: 27.000000
--------------------------------------------------
Input: 1.0
PyTorch value: 13.591409, MyScalar value: 13.591409
PyTorch grad: 13.591409, MyScalar grad: 13.591409
--------------------------------------------------
Input: 0.7
PyTorch valu